In [23]:
import os
import sys
if 'Utils.Imports' in sys.modules:
    del sys.modules['Utils.Imports']
from Utils.Imports import *
if 'Utils.Functions' in sys.modules:
    del sys.modules['Utils.Functions']
from Utils.Functions import *

In [24]:
input_cols = [
    'nose_x', 'nose_y', 'nose_z',
    'left_eye_x', 'left_eye_y', 'left_eye_z',
    'right_eye_x', 'right_eye_y', 'right_eye_z',
    'left_ear_x', 'left_ear_y', 'left_ear_z',
    'right_ear_x', 'right_ear_y', 'right_ear_z',
    'left_shoulder_x', 'left_shoulder_y', 'left_shoulder_z',
    'right_shoulder_x', 'right_shoulder_y', 'right_shoulder_z',
    'left_elbow_x', 'left_elbow_y', 'left_elbow_z',
    'right_elbow_x', 'right_elbow_y', 'right_elbow_z',
    'left_wrist_x', 'left_wrist_y', 'left_wrist_z',
    'right_wrist_x', 'right_wrist_y', 'right_wrist_z',
    'left_hip_x', 'left_hip_y', 'left_hip_z',
    'right_hip_x', 'right_hip_y', 'right_hip_z',
    'left_knee_x', 'left_knee_y', 'left_knee_z',
    'right_knee_x', 'right_knee_y', 'right_knee_z',
    'left_ankle_x', 'left_ankle_y', 'left_ankle_z',
    'right_ankle_x', 'right_ankle_y', 'right_ankle_z'
]

target_cols = [
    'elbowL_yz_deg', 'elbowR_yz_deg',
    'head_vs_neck_xz_deg', 'head_vs_neck_yz_deg',
    'kneeL_xz_deg', 'kneeL_yz_deg', 'kneeR_xz_deg', 'kneeR_yz_deg',
    'shoulderL_vs_upperarmL_xz_deg', 'shoulderL_vs_upperarmL_yz_deg',
    'shoulderR_vs_upperarmR_xz_deg', 'shoulderR_vs_upperarmR_yz_deg',
    'spine1_vs_spine2_yz_deg', 'spine2_vs_spine3_yz_deg', 'spine3_vs_spine4_yz_deg',
    'thighL_vs_spine1_yz_deg', 'thighR_vs_spine1_yz_deg',
    'wristL_yz_deg', 'wristR_yz_deg',
    'head_q_x', 'head_q_y', 'head_q_z',
    'upperarmL_q_x', 'upperarmL_q_y', 'upperarmL_q_z',
    'upperarmR_q_x', 'upperarmR_q_y', 'upperarmR_q_z',
    'handL_q_x', 'handL_q_y', 'handL_q_z',
    'handR_q_x', 'handR_q_y', 'handR_q_z',
    'thighL_q_x', 'thighL_q_y', 'thighL_q_z',
    'thighR_q_x', 'thighR_q_y', 'thighR_q_z',
    'shinL_q_x', 'shinL_q_y', 'shinL_q_z',
    'shinR_q_x', 'shinR_q_y', 'shinR_q_z',
    'spine1_q_x', 'spine1_q_y', 'spine1_q_z',
    'spine2_q_x', 'spine2_q_y', 'spine2_q_z',
    'spine3_q_x', 'spine3_q_y', 'spine3_q_z',
    'spine4_q_x', 'spine4_q_y', 'spine4_q_z'
]

In [25]:
def yolo_pose_to_custom_json(
    image_path,
    model_path="yolo11n-pose.pt",
    conf=0.25,
    save_json=False,
    json_folder=None,
    save_png=False,
    png_folder=None,

):
    """
    Ultralytics YOLO Pose ile tek görüntüden pose çıkarır ve
    sonucu meta_info + instance_info formatında döndürür.

    Not:
    - COCO pose düzeni kullanılır (17 keypoint).
    - keypoints içindeki 3. değer görünürlük/güven değeridir.
    - Gerçek 3D z koordinatı değildir.

    Parametreler
    ------------
    image_path : str
        Girdi görsel yolu
    model_path : str
        YOLO pose model yolu
    conf : float
        Güven eşiği
    save_json : bool
        JSON kaydedilsin mi
    json_folder : str | None
        JSON klasörü. json_path verilmemişse otomatik dosya adı üretilir
    json_path : str | None
        JSON tam dosya yolu
    save_png : bool
        Çizilmiş pose görseli kaydedilsin mi
    png_folder : str | None
        PNG klasörü. png_path verilmemişse otomatik dosya adı üretilir
    png_path : str | None
        PNG tam dosya yolu

    Dönenler
    --------
    result : dict
        JSON benzeri çıktı
    rendered : np.ndarray | None
        YOLO'nun çizdiği görsel
    """

    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Görüntü bulunamadı: {image_path}")

    base_name = os.path.splitext(os.path.basename(image_path))[0]

    keypoint_id2name = {
        "0": "nose",
        "1": "left_eye",
        "2": "right_eye",
        "3": "left_ear",
        "4": "right_ear",
        "5": "left_shoulder",
        "6": "right_shoulder",
        "7": "left_elbow",
        "8": "right_elbow",
        "9": "left_wrist",
        "10": "right_wrist",
        "11": "left_hip",
        "12": "right_hip",
        "13": "left_knee",
        "14": "right_knee",
        "15": "left_ankle",
        "16": "right_ankle"
    }

    keypoint_name2id = {v: int(k) for k, v in keypoint_id2name.items()}

    skeleton_links = [
        [15, 13],
        [13, 11],
        [16, 14],
        [14, 12],
        [11, 12],
        [5, 11],
        [6, 12],
        [5, 6],
        [5, 7],
        [6, 8],
        [7, 9],
        [8, 10],
        [1, 2],
        [0, 1],
        [0, 2],
        [1, 3],
        [2, 4],
        [3, 5],
        [4, 6]
    ]

    keypoint_colors = [
        [51, 153, 255], [51, 153, 255], [51, 153, 255], [51, 153, 255], [51, 153, 255],
        [0, 255, 0], [255, 128, 0], [0, 255, 0], [255, 128, 0],
        [0, 255, 0], [255, 128, 0], [0, 255, 0], [255, 128, 0],
        [0, 255, 0], [255, 128, 0], [0, 255, 0], [255, 128, 0]
    ]

    skeleton_link_colors = [
        [0, 255, 0], [0, 255, 0], [255, 128, 0], [255, 128, 0],
        [51, 153, 255], [51, 153, 255], [51, 153, 255], [51, 153, 255],
        [0, 255, 0], [255, 128, 0], [0, 255, 0], [255, 128, 0],
        [51, 153, 255], [51, 153, 255], [51, 153, 255], [51, 153, 255],
        [51, 153, 255], [51, 153, 255], [51, 153, 255]
    ]

    result = {
        "meta_info": {
            "dataset_name": "coco",
            "num_keypoints": 17,
            "keypoint_id2name": keypoint_id2name,
            "keypoint_name2id": keypoint_name2id,
            "upper_body_ids": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
            "lower_body_ids": [11, 12, 13, 14, 15, 16],
            "flip_indices": [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15],
            "flip_pairs": [
                [2, 1], [1, 2], [4, 3], [3, 4], [6, 5], [5, 6],
                [8, 7], [7, 8], [10, 9], [9, 10], [12, 11], [11, 12],
                [14, 13], [13, 14], [16, 15], [15, 16]
            ],
            "keypoint_colors": {
                "__ndarray__": keypoint_colors,
                "dtype": "uint8",
                "shape": [17, 3],
                "Corder": True
            },
            "num_skeleton_links": 19,
            "skeleton_links": skeleton_links,
            "skeleton_link_colors": {
                "__ndarray__": skeleton_link_colors,
                "dtype": "uint8",
                "shape": [19, 3],
                "Corder": True
            }
        },
        "instance_info": []
    }

    model = YOLO(model_path)

    results = model.predict(
        source=image_path,
        conf=conf,
        save=False,
        verbose=False
    )

    if not results:
        return result, None

    res = results[0]
    rendered = res.plot()

    if res.keypoints is not None and res.keypoints.data is not None:
        kp_data = res.keypoints.data.cpu().numpy()  # (N, 17, 3)

        for person_idx in range(kp_data.shape[0]):
            pts = kp_data[person_idx]

            keypoints_list = []
            keypoint_scores = []

            for kp in pts:
                x, y, v = float(kp[0]), float(kp[1]), float(kp[2])
                keypoints_list.append([x, y, v])
                keypoint_scores.append(v)

            result["instance_info"].append({
                "keypoints": keypoints_list,
                "keypoint_scores": keypoint_scores
            })

    if save_json:
       
        if json_folder is None:
            raise ValueError("save_json=True ise json_path veya json_folder verilmelidir.")
        os.makedirs(json_folder, exist_ok=True)
        json_path = os.path.join(json_folder, f"{base_name}_yolo_pose.json")

        out_dir = os.path.dirname(json_path)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)

        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        print(f"JSON kaydedildi: {json_path}")

    if save_png:
        if rendered is None:
            print("Görselleştirme oluşturulamadı, PNG kaydedilmedi.")
        else:

            if png_folder is None:
                raise ValueError("save_png=True ise png_path veya png_folder verilmelidir.")
            os.makedirs(png_folder, exist_ok=True)
            png_path = os.path.join(png_folder, f"{base_name}_yolo_pose_vis.png")
            
            cv2.imwrite(png_path, rendered)
            print(f"Görselleştirme kaydedildi: {png_path}")

    return result, rendered
def draw_pose_from_json(image_path, result):
    import cv2
    import numpy as np

    img = cv2.imread(image_path).copy()

    keypoints = result["instance_info"][0]["keypoints"]
    skeleton = result["meta_info"]["skeleton_links"]

    # Keypoint çiz
    for i, (x, y, v) in enumerate(keypoints):
        if v > 0.2:  # threshold
            cv2.circle(img, (int(x), int(y)), 4, (0,255,0), -1)

    # Kemik çiz
    for i, j in skeleton:
        x1, y1, v1 = keypoints[i]
        x2, y2, v2 = keypoints[j]

        if v1 > 0.2 and v2 > 0.2:
            cv2.line(img, (int(x1), int(y1)), (int(x2), int(y2)), (255,0,0), 2)

    return img


In [26]:
image_path = r"C:\Python_Output\Makehuman\Png\fullbody_001_000.png"

result, rendered = yolo_pose_to_custom_json(
    image_path=image_path,
    model_path="yolo11n-pose.pt",   # istersen yolo11s-pose.pt de kullanabilirsin
    conf=0.25,
    save_json=True,
    json_folder=r"C:\Python_Output\Makehuman\Tempout",
    save_png=False,
    png_folder=None,

)
picture_state=False
if picture_state:
    img_drawn = draw_pose_from_json(image_path, result)
    cv2.imshow("Custom Pose", img_drawn)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

JSON kaydedildi: C:\Python_Output\Makehuman\Tempout\fullbody_001_000_yolo_pose.json


In [30]:
meta = result.get("meta_info", {})
id2name = result.get("keypoint_id2name", {})
# 17 keypoint ismini sırayla al
kp_names = [id2name.get(str(i), f"kp{i}") for i in range(17)]

# --- keypoints (xyz)
inst_list = result.get("instance_info", [])
if inst_list:
    keypoints = inst_list[0].get("keypoints", [])
else:
    keypoints = []

if len(keypoints) < 17:
    keypoints += [[None, None, None]] * (17 - len(keypoints))
else:
    keypoints = keypoints[:17]
input_row = {}    
for i, name in enumerate(kp_names):
    x, y, z = keypoints[i]
    input_row[f"{name}_x"] = x
    input_row[f"{name}_y"] = y
    input_row[f"{name}_z"] = z
df_input = pd.DataFrame([input_row])
df_input.columns=input_cols
df_input = add_pose_angles(df_input, add_planar_angles=True)
print(df_input.columns.tolist())
input_cols=df_input.columns.tolist()

['nose_x', 'nose_y', 'nose_z', 'left_eye_x', 'left_eye_y', 'left_eye_z', 'right_eye_x', 'right_eye_y', 'right_eye_z', 'left_ear_x', 'left_ear_y', 'left_ear_z', 'right_ear_x', 'right_ear_y', 'right_ear_z', 'left_shoulder_x', 'left_shoulder_y', 'left_shoulder_z', 'right_shoulder_x', 'right_shoulder_y', 'right_shoulder_z', 'left_elbow_x', 'left_elbow_y', 'left_elbow_z', 'right_elbow_x', 'right_elbow_y', 'right_elbow_z', 'left_wrist_x', 'left_wrist_y', 'left_wrist_z', 'right_wrist_x', 'right_wrist_y', 'right_wrist_z', 'left_hip_x', 'left_hip_y', 'left_hip_z', 'right_hip_x', 'right_hip_y', 'right_hip_z', 'left_knee_x', 'left_knee_y', 'left_knee_z', 'right_knee_x', 'right_knee_y', 'right_knee_z', 'left_ankle_x', 'left_ankle_y', 'left_ankle_z', 'right_ankle_x', 'right_ankle_y', 'right_ankle_z', 'left_elbow_angle_3d', 'right_elbow_angle_3d', 'left_shoulder_angle_3d', 'right_shoulder_angle_3d', 'left_hip_angle_3d', 'right_hip_angle_3d', 'left_knee_angle_3d', 'right_knee_angle_3d', 'trunk_vs_lef

TAHMİN YAPMA

In [31]:
# =========================================================
# 0) IMPORTS
# =========================================================
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Eğer kendi yardımcı import yapını kullanıyorsan aç:
if 'Utils.Imports' in sys.modules:
    del sys.modules['Utils.Imports']
from Utils.Imports import *
if 'Utils.Models' in sys.modules:
    del sys.modules['Utils.Models']
from Utils.Models import *


# =========================================================
# 2) EĞİTİMDE KULLANILAN GİRİŞ ve ÇIKIŞ KOLONLARI
# =========================================================


n_features = len(input_cols)
n_targets = len(target_cols)
use_uncertainty = False


# =========================================================
# 3) DOSYA YOLLARI
# =========================================================
model_folder = r"C:\Python_Output\Makehuman\Model"

best_model_path = os.path.join(model_folder, "best_ft_transformer_multioutput.keras")
scaler_path     = os.path.join(model_folder, "ft_transformer_scaler.pkl")
meta_path       = os.path.join(model_folder, "advanced_ft_transformer_meta.json")

# =========================================================
# 4) SCALER ve MODELİ YÜKLE
# =========================================================
scaler = joblib.load(scaler_path)

custom_objects_dict = {
    "FeatureTokenizer": FeatureTokenizer,
    "FeaturePositionalEmbedding": FeaturePositionalEmbedding,
    "AttentionPooling": AttentionPooling,
    "ResidualMLPBlock": ResidualMLPBlock,
}

if use_uncertainty:
    custom_objects_dict["loss_fn"] = heteroscedastic_multioutput_loss(n_targets)
    custom_objects_dict["heteroscedastic_multioutput_loss"] = heteroscedastic_multioutput_loss

model = keras.models.load_model(
    best_model_path,
    custom_objects=custom_objects_dict,
    compile=False
)

print("Scaler yüklendi :", scaler_path)
print("Model yüklendi  :", best_model_path)



c:\Users\Meryem\anaconda3\Lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'feature_pos_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
c:\Users\Meryem\anaconda3\Lib\site-packages\keras\src\layers\layer.py:421: UserWarning: `build()` was called on layer 'attention_pooling', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(


Scaler yüklendi : C:\Python_Output\Makehuman\Model\ft_transformer_scaler.pkl
Model yüklendi  : C:\Python_Output\Makehuman\Model\best_ft_transformer_multioutput.keras


In [32]:
result = predict_from_df_input(
    df_input=df_input,
    model=model,
    scaler=scaler,
    input_cols=input_cols,
    target_cols=target_cols,
    use_uncertainty=use_uncertainty,
    batch_size=256
)

if use_uncertainty:
    df_result, df_pred_mean, df_pred_logvar = result
    print("Tahmin edilen ortalama değerler:")
    print(df_pred_mean.T)
    print("\nLog-variance değerleri:")
    print(df_pred_logvar.T)
else:
    df_result, df_pred = result
    print("Tahmin sonuçları:")
    print(df_pred.T)

# =========================================================
# 9) İSTERSEN SONUCU KAYDET
# =========================================================
out_csv = os.path.join(model_folder, "single_input_prediction.csv")
df_result.to_csv(out_csv, index=False, encoding="utf-8-sig")
print("\nSonuç kaydedildi:", out_csv)

c:\Users\Meryem\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Tahmin sonuçları:
                                        0
elbowL_yz_deg                  158.900391
elbowR_yz_deg                  160.172791
head_vs_neck_xz_deg              5.668374
head_vs_neck_yz_deg             17.545992
kneeL_xz_deg                   167.642349
kneeL_yz_deg                   119.897545
kneeR_xz_deg                   168.286621
kneeR_yz_deg                   120.196327
shoulderL_vs_upperarmL_xz_deg   84.368431
shoulderL_vs_upperarmL_yz_deg  127.146988
shoulderR_vs_upperarmR_xz_deg   81.924965
shoulderR_vs_upperarmR_yz_deg  137.935471
spine1_vs_spine2_yz_deg          8.930127
spine2_vs_spine3_yz_deg          4.231018
spine3_vs_spine4_yz_deg         13.915399
thighL_vs_spine1_yz_deg        125.454590
thighR_vs_spine1_yz_deg        123.811531
wristL_yz_deg                  168.056137
wristR_yz_deg                  159.065704
head_q_x                        -0.042496
head_q_y                         0.081911
head_q_z                        -0.011399
upperarmL_q_x   

REBA SKORU HESAPLAMA

In [33]:
import sys
if 'Utils.Reba' in sys.modules:
    del sys.modules['Utils.Reba']
from Utils.Reba import *

In [34]:
datamode=2
if datamode==1:
    real_csv_path = r"C:\Python_Output\Makehuman\DL_Data\df_out_target_angles.csv"
    df_reba = pd.read_csv(real_csv_path)

else:
    df_reba = df_result.copy()
df_reba=df_reba[target_cols]    
df_reba["thighR_vs_spine1_yz_from180_abs"] = (180 - df_reba["thighR_vs_spine1_yz_deg"]).abs()
df_reba["thighL_vs_spine1_yz_from180_abs"] = (180 - df_reba["thighL_vs_spine1_yz_deg"]).abs()
df_reba["kneeR_yz_deg180_abs"] = (180 - df_reba["kneeR_yz_deg"]).abs()
df_reba["kneeL_yz_deg180_abs"] = (180 - df_reba["kneeL_yz_deg"]).abs()
df_reba["kneeR_xz_deg180_abs"] = (180 - df_reba["kneeR_xz_deg"]).abs()
df_reba["kneeL_xz_deg180_abs"] = (180 - df_reba["kneeL_xz_deg"]).abs()
df_reba["upperarmL_vs_shoulderL_yz_from180_abs"] = (180 - df_reba["shoulderL_vs_upperarmL_yz_deg"]).abs()
df_reba["upperarmR_vs_shoulderR_yz_from180_abs"] = (180 - df_reba["shoulderR_vs_upperarmR_yz_deg"]).abs()


In [35]:
# =========================================================
# Kullanım
# =========================================================
df_reba_score = compute_reba_scores(df_reba, rules)

# Sadece score kolonları istenirse:
score_cols = [c for c in df_reba_score.columns if c.endswith("_score")]
df_reba_score = df_reba_score[["Score_Step1", "Score_Step2", "Score_Step3","Score_Step4", "Score_Step5", "Score_Step6"]]

# =========================================================
# Varsayılan kolonlar yoksa oluştur
# =========================================================
if "force_load_kg" not in df_reba_score.columns:
    df_reba_score["force_load_kg"] = 0

if "shock_or_rapid" not in df_reba_score.columns:
    df_reba_score["shock_or_rapid"] = False

if "coupling" not in df_reba_score.columns:
    df_reba_score["coupling"] = 0

if "static_posture" not in df_reba_score.columns:
    df_reba_score["static_posture"] = False

if "repetitive_small_range" not in df_reba_score.columns:
    df_reba_score["repetitive_small_range"] = False

if "rapid_large_change_or_unstable" not in df_reba_score.columns:
    df_reba_score["rapid_large_change_or_unstable"] = False


In [36]:
df_reba_score["TableA_score"] = df_reba_score.apply(
    lambda x: get_reba_tableA_score(
        neck=clamp(x["Score_Step1"], 1, 3),
        trunk=clamp(x["Score_Step2"], 1, 5),
        legs=clamp(x["Score_Step3"], 1, 4)
    ),
    axis=1
)
if "shock_or_rapid" not in df_reba_score.columns:
    df_reba_score["shock_or_rapid"] = False
    
df_reba_score["ForceLoad_score"] = df_reba_score.apply(
    lambda x: get_force_load_score(
        load_value=x["force_load_kg"],
        shock_or_rapid=bool(x["shock_or_rapid"])
    ),
    axis=1
)

# Resimdeki Score A = Posture Score A + Force/Load Score
df_reba_score["ScoreA"] = df_reba_score["TableA_score"] + df_reba_score["ForceLoad_score"]


In [37]:
df_reba_score["TableB_score"] = df_reba_score.apply(
    lambda x: get_reba_tableB_score(
        upper_arm=clamp(x["Score_Step4"], 1, 6),
        lower_arm=clamp(x["Score_Step5"], 1, 2),
        wrist=clamp(x["Score_Step6"], 1, 3),
    ),
    axis=1
)

# coupling kolonu yoksa varsayılan 0
if "coupling" not in df_reba_score.columns:
    df_reba_score["coupling"] = 0

df_reba_score["Coupling_score"] = df_reba_score["coupling"].apply(get_coupling_score)

# Step 12: Score B = Posture Score B + Coupling Score
df_reba_score["ScoreB"] = df_reba_score["TableB_score"] + df_reba_score["Coupling_score"]




In [38]:
df_reba_score["TableC_score"] = df_reba_score.apply(
    lambda x: get_reba_tableC_score(
        tableA=clamp(x["ScoreA"], 1, 12),
        tableB=clamp(x["ScoreB"], 1, 12)
    ),
    axis=1
)

In [39]:
# =========================================================
# STEP 13: ACTIVITY SCORE
# =========================================================
df_reba_score["Activity_score"] = df_reba_score.apply(
    lambda x: get_activity_score(
        static_posture=x["static_posture"],
        repetitive_small_range=x["repetitive_small_range"],
        rapid_large_change_or_unstable=x["rapid_large_change_or_unstable"]
    ),
    axis=1
)


In [40]:
# =========================================================
# FINAL REBA
# =========================================================
df_reba_score["Final_REBA_score"] = (
    df_reba_score["TableC_score"] + df_reba_score["Activity_score"]
)

df_reba_score["REBA_risk_level"] = df_reba_score["Final_REBA_score"].apply(classify_reba_risk)

In [41]:
cols_show = [
    "Score_Step1", "Score_Step2", "Score_Step3",
    "TableA_score", "ForceLoad_score", "ScoreA",

    "Score_Step4", "Score_Step5", "Score_Step6",
    "TableB_score", "Coupling_score", "ScoreB",

    "TableC_score", "Activity_score",
    "Final_REBA_score", "REBA_risk_level"
]

df_reba_score[cols_show].head()

,Score_Step1,Score_Step2,Score_Step3,TableA_score,ForceLoad_score,ScoreA,Score_Step4,Score_Step5,Score_Step6,TableB_score,Coupling_score,ScoreB,TableC_score,Activity_score,Final_REBA_score,REBA_risk_level
0,2,6,3,8,0,8,6,4,4,9,0,9,10,0,10,High risk
